# src\user_app\serializers.py:

In [ ]:
from django.contrib.auth.models import User
from rest_framework.serializers import ModelSerializer, ValidationError, CharField

from django.contrib.auth.models import User
from rest_framework import serializers

class UserProfileSerializer(serializers.ModelSerializer):

    class Meta:
        model = User
        fields = ["id", "username", "email"]

class RegistrationSerializer(ModelSerializer):
    password2 = CharField(
        style={"input_type": "password"}, write_only=True
    )

    class Meta:
        model = User
        fields = ["username", "email", "password", "password2"]
        extra_kwargs = {
        "password": {"write_only": True}
        }

    def save(self):
        password = self.validated_data["password"]
        password2 = self.validated_data["password2"]
        
        if password != password2:
            raise ValidationError({"error":"password y password2 deben ser iguales"})
        
        email = self.validated_data["email"]
        
        if User.objects.filter(email=email).exists():
            raise ValidationError({"error":"Esa correo ya esta registrado"})

        username = self.validated_data["username"]
        
        account = User(email=email, username=username)
        account.set_password(password)
        account.save()
        
        return account
       


# src\user_app\urls.py:

In [ ]:
from django.urls import path
from rest_framework.authtoken.views import obtain_auth_token
from rest_framework_simplejwt.views import TokenObtainPairView, TokenRefreshView

from user_app.views import registration_view, logout_view

from .views import UserProfileView

urlpatterns = [
    path("login/", obtain_auth_token),
    # path("register/", registration_view),
    path("logout/", logout_view),
    # path("api/token", TokenObtainPairView.as_view()),
    # path("api/token/refresh", TokenRefreshView.as_view()),
    
    path("token/refresh/", TokenRefreshView.as_view()),
    path("profile/", UserProfileView.as_view(), name="profile"), #<----------

]


# src\user_app\views.py:

In [ ]:
from django.shortcuts import render

from rest_framework.decorators import api_view
from rest_framework.response import Response

from user_app.serializers import RegistrationSerializer
from rest_framework.authtoken.models import Token
from rest_framework import status
from rest_framework_simplejwt.tokens import RefreshToken

# Create your views here.

from rest_framework.views import APIView #<----------
from rest_framework.response import Response #<----------
from rest_framework.permissions import IsAuthenticated #<----------

from .serializers import UserProfileSerializer #<----------

class UserProfileView(APIView): #<----------

    permission_classes = [IsAuthenticated]

    def get(self, request):

        serializer = UserProfileSerializer(request.user)

        return Response(serializer.data)
    
@api_view(["POST"])
def logout_view(request):

    return Response(
        {"response": "Logout exitoso"},
        status=status.HTTP_200_OK
    )
    
# @api_view(["POST"])
# def logout_view(request):
    
#     if request.method == "POST":
#         request.user.auth_token.delete()
#         return Response(status=status.HTTP_200_OK)

@api_view(["POST"])
def registration_view(request):
    
    if request.method == "POST":
        serializer = RegistrationSerializer(data=request.data)
        
        data = {}
        
        if serializer.is_valid():
            account = serializer.save()
            
            data["response"] = "Registro exitoso"
            data["username"] = account.username
            data["email"] = account.email
            
            # token = Token.objects.get(user=account).key
            # data["token"] = token
        
            refresh_token = RefreshToken.for_user(account) #<-------
            data["token"] = {
                "refresh":str(refresh_token),
                "access":str(refresh_token.access_token)
            }
        else:
            data = serializer.errors
    
        return Response(data)


# src\config\settings.py:

In [ ]:
……………..
INSTALLED_APPS = [
    "user_app.apps.UserAppConfig", #<----------
    "rest_framework", 
    "rest_framework.authtoken",
    "rest_examples.apps.RestExamplesConfig", 
    "api.apps.ApiConfig", 
    "test_templates.apps.TestTemplatesConfig", 
    "forms_test", 
    "cart",   
    "billing_profile",
    "address",
    "order_manage",
    "ventas",
    "accounts",
    "pages.apps.PagesConfig",
    "products.apps.ProductsConfig", 
    "ecommerce.apps.EcommerceConfig",
    "base.apps.BaseConfig",  
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
]
……………..
REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": [
        "rest_framework_simplejwt.authentication.JWTAuthentication",
    ],
    
    "DEFAULT_PERMISSION_CLASSES": [
        # "rest_framework.permissions.IsAuthenticated",
    ]
}
……………..


# src\config\urls.py:

In [ ]:
"""
URL configuration for hello project.

The `urlpatterns` list routes URLs to views. For more information please see:
    https://docs.djangoproject.com/en/6.0/topics/http/urls/
Examples:
Function views
    1. Add an import:  from my_app import views
    2. Add a URL to urlpatterns:  path('', views.home, name='home')
Class-based views
    1. Add an import:  from other_app.views import Home
    2. Add a URL to urlpatterns:  path('', Home.as_view(), name='home')
Including another URLconf
    1. Import the include() function: from django.urls import include, path
    2. Add a URL to urlpatterns:  path('blog/', include('blog.urls'))
"""

from django.conf import settings
from django.contrib import admin
from django.urls import include, path
from ventas.views import SalesChartView

from rest_framework_simplejwt.views import ( #<----------
    TokenObtainPairView,
    TokenRefreshView,
)
urlpatterns = [
    path("up/", include("up.urls")),
    path("", include("pages.urls")),
    path("ecommerce/", include("ecommerce.urls")),
    path("products/", include("products.urls")),  
    path("forms/", include("forms_test.urls")),  
    path("admin/", admin.site.urls),
    path('ventas/', include('ventas.urls')),
    path("accounts/", include("accounts.urls")), 
    path("test_templates/", include("test_templates.urls")),  
    path('productos/', include('test_templates.urls')), 
    path("api/v1/", include("api.urls")), 
    path("api/v2/", include("rest_examples.urls")), 
    
    path("api/token/", TokenObtainPairView.as_view(), name="token_obtain_pair"), #<----------
    path("api/token/refresh/", TokenRefreshView.as_view(), name="token_refresh"), #<----------
    path("account/", include("user_app.urls")), #<----------

]
if not settings.TESTING:
    urlpatterns = [
        *urlpatterns,
        path("__debug__/", include("debug_toolbar.urls")),

    ]
